In [1]:
import os
import sys
import json
import sqlite3
import struct
import msgpack
import numpy as np
from typing import Any, Dict, List, Tuple, Optional


In [2]:
def is_sqlite_file(path: str) -> bool:
    """Check whether a file is a SQLite database by header."""
    with open(path, "rb") as f:
        header = f.read(16)
    return header.startswith(b"SQLite format 3\x00")


def try_msgpack_load(path: str) -> Dict[str, Any]:
    """Load a msgpack file into a Python dict."""
    with open(path, "rb") as f:
        data = msgpack.unpackb(f.read(), raw=False, strict_map_key=False)
    if not isinstance(data, dict):
        raise ValueError(f"Msgpack root is not a dict. Got type={type(data)}")
    return data


def summarize_obj(obj: Any, max_items: int = 5, max_str: int = 200) -> Any:
    """
    Make a JSON-friendly summary of an object without exploding.
    This is for inspection only.
    """
    if obj is None:
        return None

    if isinstance(obj, (bool, int, float)):
        return obj

    if isinstance(obj, str):
        return obj if len(obj) <= max_str else obj[:max_str] + "..."

    if isinstance(obj, bytes):
        return {"__bytes__": len(obj)}

    if isinstance(obj, list):
        return {
            "__type__": "list",
            "len": len(obj),
            "sample": [summarize_obj(x, max_items=max_items) for x in obj[:max_items]],
        }

    if isinstance(obj, dict):
        keys = list(obj.keys())
        return {
            "__type__": "dict",
            "len": len(keys),
            "keys_sample": [str(k) for k in keys[:max_items]],
        }

    if isinstance(obj, np.ndarray):
        return {
            "__type__": "ndarray",
            "shape": obj.shape,
            "dtype": str(obj.dtype),
            "min": float(obj.min()) if obj.size else None,
            "max": float(obj.max()) if obj.size else None,
        }

    # Fallback for unknown types
    return {"__type__": str(type(obj))}


In [3]:
# TODO: change these paths to your actual files
MAP_PATH = "/mnt/data/UNav-IO/temp/New_York_University/Tandon/4_floor/stella_vslam_dense/final_map.msg"           # could be .msg or sqlite db in your pipeline
MAP_DENSE_PATH = "/path/to/final_map.msg_dense"  # often exists in stella_vslam_dense

KEYFRAME_DIR = "/path/to/keyframes"  # optional, only if you want to align with exported images


In [4]:
def inspect_map_file(path: str) -> Dict[str, Any]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    if is_sqlite_file(path):
        print(f"[INFO] Detected SQLite DB: {path}")
        conn = sqlite3.connect(path)
        cur = conn.cursor()

        # List tables
        cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = [r[0] for r in cur.fetchall()]
        print("[INFO] Tables:", tables)

        # For each table, show schema + row count (safe)
        info = {"format": "sqlite", "tables": {}}
        for t in tables:
            cur.execute(f"PRAGMA table_info({t})")
            schema = cur.fetchall()
            cur.execute(f"SELECT COUNT(*) FROM {t}")
            nrows = cur.fetchone()[0]
            info["tables"][t] = {
                "nrows": nrows,
                "schema": schema,  # (cid, name, type, notnull, dflt_value, pk)
            }

        conn.close()
        return info

    else:
        print(f"[INFO] Assuming msgpack map: {path}")
        msg = try_msgpack_load(path)
        print("[INFO] Top-level keys:", list(msg.keys()))
        top_summary = {k: summarize_obj(v) for k, v in msg.items()}
        return {"format": "msgpack", "top_summary": top_summary, "raw": msg}


info_map = inspect_map_file(MAP_PATH)
info_map.keys()


[INFO] Detected SQLite DB: /mnt/data/UNav-IO/temp/New_York_University/Tandon/4_floor/stella_vslam_dense/final_map.msg
[INFO] Tables: ['stats', 'cameras', 'keyframes', 'landmarks', 'associations', 'dense_points']


dict_keys(['format', 'tables'])

In [7]:
import sqlite3
import numpy as np

DB_PATH = "/mnt/data/UNav-IO/temp/New_York_University/Tandon/4_floor/stella_vslam_dense/final_map.msg"

def show_table_info(conn, table: str):
    cur = conn.cursor()
    cur.execute(f"PRAGMA table_info({table})")
    cols = cur.fetchall()  # (cid, name, type, notnull, dflt_value, pk)

    cur.execute(f"SELECT COUNT(*) FROM {table}")
    nrows = cur.fetchone()[0]

    print(f"\n=== {table} (rows={nrows}) ===")
    for cid, name, ctype, notnull, dflt, pk in cols:
        print(f"  - {name:20s} {ctype:12s} notnull={notnull} pk={pk} dflt={dflt}")

with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]
    print("[INFO] Tables:", tables)

    for t in tables:
        show_table_info(conn, t)


[INFO] Tables: ['stats', 'cameras', 'keyframes', 'landmarks', 'associations', 'dense_points']

=== stats (rows=1) ===
  - id                   INTEGER      notnull=0 pk=1 dflt=None
  - keyframe_next_id     INTEGER      notnull=0 pk=0 dflt=None
  - landmark_next_id     INTEGER      notnull=0 pk=0 dflt=None
  - dense_point_next_id  INTEGER      notnull=0 pk=0 dflt=None

=== cameras (rows=1) ===
  - id                   INTEGER      notnull=0 pk=1 dflt=None
  - name                 BLOB         notnull=0 pk=0 dflt=None
  - setup_type           BLOB         notnull=0 pk=0 dflt=None
  - model_type           BLOB         notnull=0 pk=0 dflt=None
  - color_type           BLOB         notnull=0 pk=0 dflt=None
  - cols                 INTEGER      notnull=0 pk=0 dflt=None
  - rows                 INTEGER      notnull=0 pk=0 dflt=None
  - fps                  REAL         notnull=0 pk=0 dflt=None
  - fx                   REAL         notnull=0 pk=0 dflt=None
  - fy                   REAL        

In [8]:
import pandas as pd

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query("SELECT * FROM dense_points LIMIT 5", conn)

df


,id,pos_w,color,ref_keyfrm
0,0,b'\xbdK\xdb\xf3/&\xc0?\xe2\x91@\xb1#?\xef\xbf\...,b'\x1b1H',3
1,1,b'e=yp\x1c\x17r\xbfA\xf8\x10\xe4\x94/\xe8\xbf6...,b'\x1a8]',3
2,2,b'uV\x1b\x9aI\xda\xb3?\xff&\xf3n\x96\x96\xc7\x...,b'\x193E',3
3,3,b'7\x1a\xf1\xc4\x9b\xe5\xb3?\xdc4\x12\xfau\xb3...,b'\x192F',3
4,4,b'\xeb\xc9\xf5\x05\xe3\xc2\x88\xbfN\x00\xdd9\x...,b'\x1f9[',3


In [9]:
def try_parse_blob(blob: bytes):
    """
    Try to parse a blob into a numpy array using common dtypes.
    Returns a dict of possible parses (dtype -> array preview).
    """
    if blob is None:
        return None
    if not isinstance(blob, (bytes, bytearray)):
        return {"not_blob": str(type(blob))}

    out = {"nbytes": len(blob), "candidates": {}}
    for dt in [np.float64, np.float32, np.int64, np.int32, np.uint8]:
        if len(blob) % np.dtype(dt).itemsize != 0:
            continue
        arr = np.frombuffer(blob, dtype=dt)
        # Only show first few values
        out["candidates"][str(dt)] = {
            "len": int(arr.size),
            "head": arr[:10].tolist(),
        }
    return out

with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()
    cur.execute("SELECT * FROM dense_points LIMIT 1")
    row = cur.fetchone()
    colnames = [d[0] for d in cur.description]

print("[INFO] dense_points columns:", colnames)
for name, val in zip(colnames, row):
    if isinstance(val, (bytes, bytearray)):
        print(f"\n--- {name} (BLOB) ---")
        print(try_parse_blob(val))
    else:
        print(f"{name}: {val}")


[INFO] dense_points columns: ['id', 'pos_w', 'color', 'ref_keyfrm']
id: 0

--- pos_w (BLOB) ---
{'nbytes': 24, 'candidates': {"<class 'numpy.float64'>": {'len': 3, 'head': [0.12616538435994853, -0.976457449146036, -0.024523309467012918]}, "<class 'numpy.float32'>": {'len': 6, 'head': [-3.4748814906680796e+31, 1.5011652708053589, -2.8022602016619658e-09, -1.8691142797470093, 3.8158228560968065e+29, -1.1961864233016968]}, "<class 'numpy.int64'>": {'len': 3, 'head': [4593713607314459581, -4616401670501264926, -4640646452734189905]}, "<class 'numpy.int32'>": {'len': 6, 'head': [-203732035, 1069557295, -1321168414, -1074839773, 1889148591, -1080484701]}, "<class 'numpy.uint8'>": {'len': 24, 'head': [189, 75, 219, 243, 47, 38, 192, 63, 226, 145]}}}

--- color (BLOB) ---
{'nbytes': 3, 'candidates': {"<class 'numpy.uint8'>": {'len': 3, 'head': [27, 49, 72]}}}
ref_keyfrm: 3


In [10]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()
    cur.execute("PRAGMA table_info(dense_points)")
    cols = [r[1] for r in cur.fetchall()]
cols


['id', 'pos_w', 'color', 'ref_keyfrm']

In [14]:
import sqlite3
import numpy as np
import pandas as pd

DB_PATH = "/mnt/data/UNav-IO/temp/New_York_University/Tandon/4_floor/stella_vslam_dense/final_map.msg"
W, H = 5760, 2880  # Your equirectangular image size

def load_keyframe_Tcw(conn, kf_id: int) -> np.ndarray:
    """Load 4x4 T_cw from keyframes table."""
    cur = conn.cursor()
    cur.execute("SELECT pose_cw FROM keyframes WHERE id=?", (kf_id,))
    pose_blob = cur.fetchone()[0]
    try:
        T_cw = np.frombuffer(pose_blob, dtype=np.float64).reshape(4, 4)
    except ValueError:
        T_cw = np.frombuffer(pose_blob, dtype=np.float32).reshape(4, 4)
    return T_cw.astype(np.float64)

def project_equirectangular(T_cw: np.ndarray, Pw: np.ndarray, W: int, H: int):
    """
    Project world points Pw (N,3) to equirectangular pixel coords (u,v).

    Returns:
        uv: (N,2) float64
        valid: (N,) bool   (points with non-zero norm and finite)
        dir_cam: (N,3) float64 normalized direction in camera frame
    """
    N = Pw.shape[0]
    Pw_h = np.hstack([Pw, np.ones((N, 1), dtype=np.float64)])     # (N,4)
    Pc_h = (T_cw @ Pw_h.T).T                                      # (N,4)
    Pc = Pc_h[:, :3]                                              # (N,3)

    # Normalize direction
    norm = np.linalg.norm(Pc, axis=1) + 1e-12
    valid = np.isfinite(norm) & (norm > 1e-9)
    d = np.zeros_like(Pc)
    d[valid] = Pc[valid] / norm[valid][:, None]

    # lon in (-pi, pi], lat in [-pi/2, pi/2]
    lon = np.arctan2(d[:, 0], d[:, 2])
    lat = np.arcsin(np.clip(d[:, 1], -1.0, 1.0))

    u = (lon / (2.0 * np.pi) + 0.5) * W
    v = (0.5 - lat / np.pi) * H

    # Wrap u, clamp v
    u = np.mod(u, W)
    v = np.clip(v, 0, H - 1)

    uv = np.stack([u, v], axis=1)
    return uv, valid, d


In [15]:
SAMPLE_N = 5000

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(
        f"SELECT id, pos_w, ref_keyfrm FROM dense_points ORDER BY RANDOM() LIMIT {SAMPLE_N}",
        conn
    )

Pw = np.vstack([np.frombuffer(b, dtype=np.float64) for b in df["pos_w"].values])
ref_kfs = df["ref_keyfrm"].values.astype(int)

uv_all = np.zeros((SAMPLE_N, 2), dtype=np.float64)
valid_all = np.zeros((SAMPLE_N,), dtype=bool)

with sqlite3.connect(DB_PATH) as conn:
    for kf_id in np.unique(ref_kfs):
        idx = np.where(ref_kfs == kf_id)[0]
        T_cw = load_keyframe_Tcw(conn, int(kf_id))
        uv, valid, _ = project_equirectangular(T_cw, Pw[idx], W, H)
        uv_all[idx] = uv
        valid_all[idx] = valid

print("[INFO] valid projections:", int(valid_all.sum()), "/", SAMPLE_N)
print("[INFO] u range:", float(np.nanmin(uv_all[:,0])), float(np.nanmax(uv_all[:,0])))
print("[INFO] v range:", float(np.nanmin(uv_all[:,1])), float(np.nanmax(uv_all[:,1])))

df_out = pd.DataFrame({
    "dense_id": df["id"].values,
    "ref_keyfrm": ref_kfs,
    "u": uv_all[:,0],
    "v": uv_all[:,1],
    "valid": valid_all
})
df_out.head()


[INFO] valid projections: 5000 / 5000
[INFO] u range: 0.3663609125404932 5759.469514774062
[INFO] v range: 305.97211510125976 2729.9420798049478


,dense_id,ref_keyfrm,u,v,valid
0,1624762,533,2153.564734,1237.134045,True
1,1263201,408,1240.146552,1354.854700,True
2,492582,184,2139.321209,1319.873317,True
3,62110,35,3232.951437,1468.360838,True
4,30440,26,3338.957992,1966.111657,True
